pressure over time

Reads the daymond data variablep, samples the timesteps you configure, and writes PSV.


Adjust frames, face, and quality in the params cell to match needs.

In [1]:
import numpy as np
import OpenVisus as ov
from pathlib import Path
import pyvista as pv
from tqdm import tqdm

In [ ]:
# --- params: edit these ---
variable = "p"  # pressure field in daymond dataset
face = 2
quality = -8  # OpenVisus resolution level (e.g. 1/8 downsample)

# How many output frames and which dataset time indices to read (0 … 10268 inclusive in this idx)
num_frames = 500
time_max = 10268
frames = np.linspace(0, time_max, num_frames, dtype=int).tolist()

# Output: collection file + one VTI per timestep (same folder)
out_dir = Path("pressure_over_time_out")
pvd_path = Path("pressure_over_time.pvd")
scalar_name = "pressure"

out_dir.mkdir(parents=True, exist_ok=True)

In [3]:
idx_url = (
    f"https://nsdf-climate3-origin.nationalresearchplatform.org:50098/"
    f"nasa/nsdf/climate3/dyamond/GEOS/GEOS_{variable.upper()}/"
    f"{variable.lower()}_face_{face}_depth_52_time_0_10269.idx"
)
db = ov.LoadDataset(idx_url)

data = db.read(time=0, quality=quality)
Z, Y, X = data.shape
grid = pv.ImageData()
grid.dimensions = (X, Y, Z)
grid.origin = (0, 0, 0)
grid.spacing = (1, 1, 20)
print(f"Field {variable}, shape (Z,Y,X)= {data.shape}, idx: {idx_url}")

Field p, shape (Z,Y,X)= (7, 180, 360), idx: https://nsdf-climate3-origin.nationalresearchplatform.org:50098/nasa/nsdf/climate3/dyamond/GEOS/GEOS_P/p_face_2_depth_52_time_0_10269.idx


In [4]:
for i, t in enumerate(tqdm(frames, desc="pressure frames")):
    arr = db.read(time=int(t), quality=quality)
    flat = arr.flatten(order="C")
    grid[scalar_name] = flat
    grid.save(out_dir / f"frame_{i:04d}.vti")

print("Done writing VTI frames.")

pressure frames: 100%|███████████████████████████████████████████████████████████████| 500/500 [04:40<00:00,  1.78it/s]

Done writing VTI frames.


In [ ]:
# PVD: timestep attribute = dataset time index (matches OpenVisus `time=`)
rel = out_dir.name
with open(pvd_path, "w", encoding="utf-8") as f:
    f.write('<?xml version="1.0"?>\n')
    f.write('<VTKFile type="Collection" version="0.1">\n')
    f.write("  <Collection>\n")
    for i, t in enumerate(frames):
        fname = f"{rel}/frame_{i:04d}.vti"
        f.write(f'    <DataSet timestep="{float(t)}" file="{fname}"/>\n')
    f.write("  </Collection>\n")
    f.write("</VTKFile>\n")

print(f"Wrote {pvd_path.resolve()} — ready for Paraview")

Wrote C:\Users\tjhes\OneDrive\Desktop\VisScientificDataTeam\SciVis-Project\pressure_and_temp\pressure_over_time.pvd — open this file in ParaView.
